# 🔺 Delta Lake — Cloning

---

## 1. O que é Cloning?

O Delta Lake permite criar **cópias de tabelas** de duas formas distintas, com comportamentos muito diferentes em relação aos dados físicos e à dependência da tabela original.

```
  Tabela source
  ├── _delta_log/   (metadados, schema, constraints, histórico)
  └── .parquet(s)   (dados físicos)
             │
     ┌───────┴───────┐
     │               │
  DEEP CLONE     SHALLOW CLONE
  Copia tudo     Copia apenas metadados
  Independente   Depende do source
```

| | Deep Clone | Shallow Clone |
|---|---|---|
| Copia metadados e schema | ✅ | ✅ |
| Copia constraints | ✅ | ✅ |
| Copia histórico (`_delta_log`) | ✅ | ✅ |
| Copia arquivos Parquet físicos | ✅ | ❌ |
| Independente do source | ✅ | ❌ |
| Custo de storage | Alto | Baixo |
| Uso recomendado | Backup, migração, isolamento | Testes, desenvolvimento, queries rápidas |

---

## 2. Tabela Source

Criamos a tabela de origem com um histórico de operações variadas — insert, delete, update e constraint —  
para demonstrar que o clone captura **todo o estado** da tabela, não apenas os dados finais.

In [ ]:
%%sql
DROP TABLE IF EXISTS source;
CREATE OR REPLACE TABLE source (id INT, content STRING);

INSERT INTO source VALUES (1, 'a'), (2, 'b'), (3, 'c'), (4, 'd');
INSERT INTO source VALUES (5, 'e');
DELETE FROM source WHERE id = 5;
UPDATE source SET content = 'f' WHERE id = 4;

ALTER TABLE source ADD CONSTRAINT Ids CHECK (id > 0);

> **Estado final da tabela source:**
> ```
>   id=1 content='a'
>   id=2 content='b'
>   id=3 content='c'
>   id=4 content='f'   ← atualizado
>   -- id=5 deletado
> ```
> O `_delta_log` registrou todos esses passos. O clone vai herdar esse histórico.

---

## 3. Deep Clone

O `DEEP CLONE` cria uma **cópia completa e independente** da tabela:  
copia o `_delta_log`, o schema, as constraints e todos os arquivos Parquet físicos para um novo local.

```
  source/
  ├── _delta_log/  ──COPIA──►  deep_clone/_delta_log/
  ├── part-001.parquet ──►    deep_clone/part-001.parquet
  └── part-002.parquet ──►    deep_clone/part-002.parquet

  Após o clone: as duas tabelas são completamente independentes.
```

In [ ]:
%%sql
DROP TABLE IF EXISTS deep_clone;

CREATE OR REPLACE TABLE deep_clone
	DEEP CLONE source;

### Inspecionando o Deep Clone

In [ ]:
%%sql
-- O histórico de operações do source é herdado pelo deep_clone
DESC HISTORY deep_clone;

In [ ]:
%%sql
-- Em 'Table Properties' podemos confirmar que a constraint 'Ids' também foi clonada
DESC DETAIL deep_clone;

In [ ]:
-- Os arquivos Parquet físicos foram copiados para o diretório do deep_clone
%fs
ls dbfs:/user/hive/warehouse/deep_clone

### Demonstrando independência do source

**Teste 1 — Novos dados na source não afetam o deep_clone:**

In [ ]:
%%sql
INSERT INTO source VALUES (5, 'New data in source table');

In [ ]:
%%sql
-- O deep_clone reflete apenas o snapshot do momento do clone — id=5 não aparece
SELECT * FROM deep_clone;

**Teste 2 — Destruir a source (inclusive os Parquets via VACUUM) não afeta o deep_clone:**

In [ ]:
%%sql
DELETE FROM source;
SET spark.databricks.delta.retentionDurationCheck.enabled = False;

In [ ]:
%%sql
VACUUM source RETAIN 0 HOURS;

In [ ]:
%%sql
-- ✅ Dados intactos — o deep_clone tem suas próprias cópias físicas dos Parquets
SELECT * FROM deep_clone;

> **Conclusão Deep Clone:**  
> Uma vez criado, o `deep_clone` é completamente autônomo.  
> Qualquer modificação, deleção ou limpeza na tabela source não tem nenhum efeito sobre ele.

---

## 4. Recriando a tabela source

Recriamos a source do zero para os testes do Shallow Clone.

In [ ]:
%%sql
DROP TABLE IF EXISTS source;
CREATE OR REPLACE TABLE source (id INT, content STRING);

INSERT INTO source VALUES (1, 'a'), (2, 'b'), (3, 'c'), (4, 'd');
INSERT INTO source VALUES (5, 'e');
DELETE FROM source WHERE id = 5;
UPDATE source SET content = 'f' WHERE id = 4;

ALTER TABLE source ADD CONSTRAINT Ids CHECK (id > 0);

---
## 5. Shallow Clone

O `SHALLOW CLONE` copia apenas os **metadados** (schema, constraints, `_delta_log`).  
Os arquivos Parquet físicos **não são copiados** — o clone aponta diretamente para os arquivos da source.

```
  source/
  ├── _delta_log/  ──COPIA──►  shallow_clone/_delta_log/
  ├── part-001.parquet ────────────────────────────────┐
  └── part-002.parquet ──────────────────────────────┐ │
                                                     │ │
  shallow_clone/                                     │ │
  ├── _delta_log/                                    │ │
  └── (sem Parquets próprios — aponta para source) ◄─┘─┘

  Se a source for destruída: shallow_clone perde os dados.
```

In [ ]:
%%sql
DROP TABLE IF EXISTS shallow_clone;

CREATE OR REPLACE TABLE shallow_clone
	SHALLOW CLONE source;

### Inspecionando o Shallow Clone

In [ ]:
%%sql
-- Histórico herdado da source, assim como no deep clone
DESC HISTORY shallow_clone;

In [ ]:
%%sql
-- Constraints também herdadas
DESC DETAIL shallow_clone;

In [ ]:
-- ⚠️ Diretório vazio (ou apenas _delta_log) — nenhum Parquet foi copiado
-- O shallow clone 'empresta' os arquivos físicos da source
%fs
ls dbfs:/user/hive/warehouse/shallow_clone

### Novos dados na source não aparecem no shallow_clone

Apesar de depender fisicamente da source, o shallow clone é um **snapshot** — não um espelho em tempo real.

In [ ]:
%%sql
INSERT INTO source VALUES (5, 'New data in source table');

In [ ]:
%%sql
-- id=5 não aparece — o clone reflete apenas o estado do momento da criação
SELECT * FROM shallow_clone;

### ⚠️ O perigo do Shallow Clone — destruindo a source

Aqui está a diferença crítica: como o shallow clone **não tem cópias próprias** dos Parquets,  
destruir os arquivos físicos da source torna os dados do clone **irrecuperáveis**.

In [ ]:
%%sql
DELETE FROM source;
SET spark.databricks.delta.retentionDurationCheck.enabled = False;

In [ ]:
%%sql
VACUUM source RETAIN 0 HOURS;

In [ ]:
%%sql
-- ❌ Erro: os arquivos Parquet que o shallow_clone referencia não existem mais no disco
SELECT * FROM shallow_clone;

> **Por que o erro ocorre?**
> ```
>   shallow_clone/_delta_log  →  aponta para  →  source/part-001.parquet
>                                                         ↑
>                                              VACUUM deletou do disco
>                                                         ↓
>                                               ❌ FileNotFoundException
> ```
> O `_delta_log` do shallow clone ainda existe e registra os arquivos corretos,  
> mas os arquivos físicos referenciados foram removidos da source pelo `VACUUM`.

---

## 6. Resumo — Deep Clone vs Shallow Clone

```
┌──────────────────────────────────────────────────────────────────┐
│                    DEEP CLONE                                    │
│                                                                  │
│  + Cópia completa: metadados + Parquets físicos                  │
│  + Totalmente independente da source                             │
│  + Sobrevive a VACUUM e DROP da source                           │
│  - Custo de storage: duplica o tamanho dos dados                 │
│  - Tempo de criação proporcional ao volume de dados              │
│  → Ideal para: backups, migração, ambiente de produção isolado   │
├──────────────────────────────────────────────────────────────────┤
│                    SHALLOW CLONE                                 │
│                                                                  │
│  + Criação instantânea — copia apenas metadados                  │
│  + Custo de storage próximo de zero                              │
│  - Depende dos Parquets físicos da source                        │
│  - VACUUM na source pode quebrar o clone                         │
│  → Ideal para: desenvolvimento, testes, queries exploratórias    │
└──────────────────────────────────────────────────────────────────┘

  Em ambos os casos:
  ✅ Schema herdado
  ✅ Constraints herdadas
  ✅ Histórico (_delta_log) herdado
  ✅ Snapshot — alterações futuras na source NÃO refletem no clone
```